In [3]:
#@title Imports

import random
import os
import sys
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import rasterio

In [4]:
#@title Mount Goole Drive

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}

mount_drive = True  #@param {type:"boolean"}

if mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from config import CFG
    cfg = CFG()

    cfg.ROOT = Path(root_path)
else:
    cfg = CFG()

    sys.path.append(os.path.join(cfg.ROOT))

Mounted at /content/drive


In [5]:
print(cfg.ROOT)
print(cfg.DATA_PATH)
print(type(cfg.ZIP_PATH))

/content/drive/MyDrive/MSc/Flood-Mapping
/content/drive/MyDrive/MSc/Flood-Mapping/Dataset
<class 'pathlib.PosixPath'>


In [6]:
#@title Download Dataset

def download_and_extract(cfg):
    data_path = cfg.DATA_PATH
    zip_path = cfg.ZIP_PATH
    sentinel1_dir = cfg.S1_PATH

    # 2. Download if not already present
    if not zip_path.exists() and not sentinel1_dir.exists():
        print("⬇️ Downloading dataset...")
        os.system(f"wget -O '{zip_path}' '{cfg.ZIP_URL}'")
    else:
        print("✅ Zip or Dataset already exists, skipping download.")

    # 3. Check if already extracted

    if sentinel1_dir.exists():
        print("✅ Dataset already extracted, skipping unzip.")
    else:
        print("📦 Extracting dataset...")

        extract_path = cfg.ROOT.resolve()  # forces correct absolute path

        print(f"Extracting to: {extract_path}")

        with zipfile.ZipFile(cfg.ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

        print("✅ Extraction complete.")

    # 4. Delete zip to save space
    if zip_path.exists():
        zip_path.unlink()
        print("🗑️ Zip file deleted.")

    # 5. Final check
    print("\n📂 Final structure:")
    for p in data_path.iterdir():
        print(" -", p.name)

    return data_path


data_root = download_and_extract(cfg)

img_dir = cfg.DATA_PATH / cfg.S1_IMAGE_PATH
mask_dir = cfg.DATA_PATH / cfg.S1_MASK_PATH

print("Image dir exists:", img_dir.exists())
print("Mask dir exists:", mask_dir.exists())

print("Num images:", len(list(img_dir.glob("*.tif"))))
print("Num masks:", len(list(mask_dir.glob("*.tif"))))


✅ Zip or Dataset already exists, skipping download.
✅ Dataset already extracted, skipping unzip.

📂 Final structure:
 - Sentinel1
 - Sentinel1_metadata.csv
 - Sentinel2
 - Sentinel2_metadata.csv
Image dir exists: True
Mask dir exists: True
Num images: 21602
Num masks: 21602


In [7]:
#@title Utility Functions

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_data_dirs(cfg):
    root = Path(cfg.DATA_PATH)
    img_dir = root / cfg.S1_IMAGE_PATH
    mask_dir = root / cfg.S1_MASK_PATH
    return img_dir, mask_dir

def build_s1_index(cfg):
    img_dir, mask_dir = get_data_dirs(cfg)

    image_paths = sorted(img_dir.glob("*.tif"))
    mask_paths = sorted(mask_dir.glob("*.tif"))

    mask_map = {p.name: p for p in mask_paths}

    pairs = []
    for img_path in image_paths:
        mask_path = mask_map.get(img_path.name)
        if mask_path is None:
            continue

        sample_id = img_path.stem
        event_id = sample_id.split("_")[0]

        pairs.append({
            "id": sample_id,
            "event_id": event_id,
            "image_path": img_path,
            "mask_path": mask_path,
        })

    if cfg.LIMIT_SAMPLES is not None:
        pairs = pairs[:cfg.LIMIT_SAMPLES]

    return pairs

def split_pairs(pairs, cfg):
    rng = random.Random(cfg.RANDOM_SEED)
    pairs = pairs.copy()
    rng.shuffle(pairs)

    n = len(pairs)
    n_train = int(n * cfg.TRAIN_RATIO)
    n_val = int(n * cfg.VAL_RATIO)

    train_pairs = pairs[:n_train]
    val_pairs = pairs[n_train:n_train + n_val]
    test_pairs = pairs[n_train + n_val:]

    return train_pairs, val_pairs, test_pairs

def read_image_tif(path):
    with rasterio.open(path) as src:
        arr = src.read()  # [C, H, W]
    return arr.astype(np.float32)

def read_mask_tif(path):
    with rasterio.open(path) as src:
        arr = src.read(1)  # [H, W]
    return arr.astype(np.int64)

def minmax_normalise_s1(image, min_db=-30.0, max_db=10.0):
    image = np.clip(image, min_db, max_db)
    image = (image - min_db) / (max_db - min_db)
    return image.astype(np.float32)

def remap_mask_to_binary(mask, water_classes=(1,2,3,4,5), ignore_classes=(99,)):
    out = np.zeros_like(mask, dtype=np.int64)

    for c in water_classes:
        out[mask == c] = 1

    for c in ignore_classes:
        out[mask == c] = 255  # common ignore index

    return out

from collections import defaultdict

def group_by_event(pairs):
    groups = defaultdict(list)

    for p in pairs:
        event_id = p["id"].split("_")[0]
        groups[event_id].append(p)

    return groups

def split_by_event(pairs, cfg):
    # 1. Group tiles by event
    groups = defaultdict(list)
    for p in pairs:
        event_id = p["event_id"] if "event_id" in p else p["id"].split("_")[0]
        groups[event_id].append(p)

    event_ids = list(groups.keys())

    # 2. Shuffle events (reproducible)
    rng = random.Random(cfg.RANDOM_SEED)
    rng.shuffle(event_ids)

    # 3. Fixed split
    n_train = cfg.N_TRAIN_EVENTS
    n_val = cfg.N_VAL_EVENTS
    n_test = cfg.N_TEST_EVENTS

    assert len(event_ids) == (n_train + n_val + n_test), \
        f"Expected 47 events, got {len(event_ids)}"

    train_events = event_ids[:n_train]
    val_events   = event_ids[n_train:n_train + n_val]
    test_events  = event_ids[n_train + n_val:]

    # 4. Flatten back to tile-level
    train_pairs = [p for e in train_events for p in groups[e]]
    val_pairs   = [p for e in val_events   for p in groups[e]]
    test_pairs  = [p for e in test_events  for p in groups[e]]

    return train_pairs, val_pairs, test_pairs

In [8]:
#@title Dataset

class SturmS1Dataset(Dataset):
    def __init__(self, pairs, cfg, is_train=False):
        self.pairs = pairs
        self.cfg = cfg
        self.is_train = is_train

    def get_events(self):
      return sorted({p["event_id"] for p in self.pairs})

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]

        image = read_image_tif(item["image_path"])   # [C,H,W]
        mask = read_mask_tif(item["mask_path"])      # [H,W]

        # sanity checks
        assert image.shape[0] == 2, f"Expected 2 channels, got {image.shape}"
        assert image.shape[1] == self.cfg.IMAGE_SIZE and image.shape[2] == self.cfg.IMAGE_SIZE
        assert mask.shape[0] == self.cfg.IMAGE_SIZE and mask.shape[1] == self.cfg.IMAGE_SIZE

        image = minmax_normalise_s1(
            image,
            min_db=self.cfg.S1_MIN_DB,
            max_db=self.cfg.S1_MAX_DB
        )


        if self.cfg.BINARY_MASK:
            mask = remap_mask_to_binary(
                mask,
                water_classes=self.cfg.WATER_CLASSES,
                ignore_classes=self.cfg.IGNORE_CLASSES
            )

        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).long()

        return {
            "image": image,
            "mask": mask,
            "id": item["id"],
        }

In [9]:
def make_dataloaders(cfg):
    set_seed(cfg.RANDOM_SEED)

    pairs = build_s1_index(cfg)
    train_pairs, val_pairs, test_pairs = split_by_event(pairs, cfg)

    train_ds = SturmS1Dataset(train_pairs, cfg, is_train=True)
    val_ds = SturmS1Dataset(val_pairs, cfg, is_train=False)
    test_ds = SturmS1Dataset(test_pairs, cfg, is_train=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=cfg.SHUFFLE_TRAIN,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
    )

    return train_loader, val_loader, test_loader

In [10]:
train_loader, val_loader, test_loader = make_dataloaders(cfg)

batch = next(iter(train_loader))
print(batch["image"].shape)   # [B, 2, 128, 128]
print(batch["mask"].shape)    # [B, 128, 128]
print(batch["image"].dtype, batch["mask"].dtype)
print(batch["id"][:3])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([16, 2, 128, 128])
torch.Size([16, 128, 128])
torch.float32 torch.int64
['EMSR429_AOI03_05_06_1_2', 'EMSR517_AOI05_11_08_2_2', 'EMSR352_AOI01_090_024_2_2']


In [11]:

train_events = train_loader.dataset.get_events()
val_events   = val_loader.dataset.get_events()
test_events  = test_loader.dataset.get_events()

print("Train events:", len(train_events))
print("Val events:", len(val_events))
print("Test events:", len(test_events))

Train events: 37
Val events: 5
Test events: 5


In [12]:
train_events = set(train_loader.dataset.get_events())
val_events = set(val_loader.dataset.get_events())
test_events = set(test_loader.dataset.get_events())

print(train_events & val_events)
print(train_events & test_events)
print(val_events & test_events)

set()
set()
set()
